# 取り込みのパターン: 5つの入力元を1つのテーブルへ

現場のデータが1か所から来ることはまずありません。ティックフィードは Arrow バッチを
渡してきますし、リサーチ用のノートブックは pandas か polars の中にあり、ベンダーは
Parquet を置いていき、古い業務プロセスはいまだに CSV をメールで送ってきます。h5i-db の
取り込み口は意図的に小さく、`append`（フィードを伸ばす）と `write`（中身を差し替える）
の2つだけで、どちらも Arrow の形をしたものなら受け取ります。このレシピでは、連続する
5営業日ぶんを5種類の入力元から1つの `trades` テーブルに取り込み、そのうえで取り込みの
運用面を扱います。`write` と `append` の違い、`expected_version` による楽観ロック、
そしてコミットをまとめてから `compact` する理由です。

In [1]:
import shutil
from pathlib import Path

import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

import h5i_db
import cookbook_utils as cu

db = h5i_db.Database(cu.fresh_db("00_ingestion"), create=True)

SCHEMA = pa.schema(
    [
        pa.field("ts", pa.timestamp("us", tz="UTC"), nullable=False),
        pa.field("symbol", pa.string()),
        pa.field("price", pa.float64()),
        pa.field("size", pa.int64()),
        pa.field("exchange", pa.string()),
        pa.field("side", pa.string()),
    ]
)
db.create_table("trades", SCHEMA, time_column="ts", sort_key=["ts", "symbol"])

{'table': 'trades',
 'sequence': 0,
 'op': 'create',
 'rows_total': 0,
 'segments_total': 0,
 'segments_added': 0,
 'segments_deduped': 0,
 'committed_at_ns': 1785040035608569055}

11セッションぶんの連続したテープを、1日ずつのバッチに切り分けます。こうすると各
「納品」が時刻順に届きます。`append` はどのバッチもテーブルの保存済み最大タイム
スタンプ以降から始まることを求めるためです（フィードとしての意味論）。ベンダーの
受け渡し場所の代わりに、`data/dbs` の下のステージング用ディレクトリを使います。

In [2]:
tape = cu.make_trades(days=11, trades_per_day=4_000, start="2026-06-01", seed=7)

dates = tape["ts"].to_pandas().dt.date
sessions = sorted(dates.unique())
by_day = {d: tape.filter(pa.array((dates == d).to_numpy())) for d in sessions}
print(f"{len(sessions)} sessions, {len(tape):,} trades:", sessions[0], "→", sessions[-1])

staging = Path("data/dbs/00_ingestion_staging")
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)

11 sessions, 141,301 trades: 2026-06-01 → 2026-06-15


## 1. pyarrow Table から

変換を挟まないネイティブの経路です。`append` は必ず**コミット辞書**を返します。中身は
新しいバージョン番号（`sequence`）と、コミット後の総行数・総セグメント数です。ローダーの
ログに残しておきましょう。納品とバージョンを結びつける受領証になります。

In [3]:
commit = db.append("trades", by_day[sessions[0]], note="day 1: arrow feed")
commit

{'table': 'trades',
 'sequence': 1,
 'op': 'append',
 'rows_total': 11418,
 'segments_total': 1,
 'segments_added': 1,
 'segments_deduped': 0,
 'committed_at_ns': 1785040035808577811}

## 2. pandas DataFrame から

重い仕事は `pa.Table.from_pandas` がやってくれますが、`schema=` は必ず渡してください。
pandas のバージョンによっては datetime がナノ秒（テーブル側はマイクロ秒）のまま往復し、
厳格な append がその不一致を拒みます。目的のスキーマを渡せば、変換のついでにキャストが
かかります。

In [4]:
df = by_day[sessions[1]].to_pandas()  # pretend this came from research code
commit = db.append(
    "trades",
    pa.Table.from_pandas(df, schema=SCHEMA, preserve_index=False),
    note="day 2: pandas",
)
{k: commit[k] for k in ("sequence", "rows_total", "segments_total")}

{'sequence': 2, 'rows_total': 25773, 'segments_total': 2}

## 3. polars DataFrame から

Polars は Arrow をそのまま話しますが、1つだけ癖があります。`to_arrow()` が出すのは
`large_string` の列で、厳格な append はこれをスキーマ不一致として拒みます。
`.cast(SCHEMA)` はメタデータ層の安い修正なので、polars から h5i-db へ渡す境界では
習慣にしてしまうのがよいでしょう。

In [5]:
pldf = pl.from_arrow(by_day[sessions[2]])  # pretend this came from a polars pipeline
commit = db.append("trades", pldf.to_arrow().cast(SCHEMA), note="day 3: polars")
{k: commit[k] for k in ("sequence", "rows_total", "segments_total")}

{'sequence': 3, 'rows_total': 38842, 'segments_total': 3}

## 4. Parquet ファイルから

Parquet は型をそのまま保つので、ベンダーが置いていった Parquet は読んで append する
だけです。（行の順序が怪しいベンダーなら、append の前にソートしてください。どちらに
しても厳格な append が教えてくれます。）

In [6]:
pq.write_table(by_day[sessions[3]], staging / "vendor_day4.parquet")

commit = db.append("trades", pq.read_table(staging / "vendor_day4.parquet"), note="day 4: parquet drop")
{k: commit[k] for k in ("sequence", "rows_total", "segments_total")}

{'sequence': 4, 'rows_total': 49876, 'segments_total': 4}

## 5. CSV から

CSV は値を保ちますが型を落とします。素直に読むと、`ts` はパーサーが推測した何かに
なって返ってきます。`ConvertOptions(column_types=...)` で時刻列をパース時点で
`timestamp[us, tz=UTC]` に固定すれば、スキーマがぴたりと一致します。

In [7]:
pacsv.write_csv(by_day[sessions[4]], staging / "legacy_day5.csv")

from_csv = pacsv.read_csv(
    staging / "legacy_day5.csv",
    convert_options=pacsv.ConvertOptions(column_types={"ts": pa.timestamp("us", tz="UTC")}),
)
assert from_csv.schema.equals(by_day[sessions[4]].schema)
commit = db.append("trades", from_csv, note="day 5: legacy csv")
{k: commit[k] for k in ("sequence", "rows_total", "segments_total")}

{'sequence': 5, 'rows_total': 63593, 'segments_total': 5}

## `write` と `append` の違い

- **`append`** はフィードを伸ばします。厳密に時刻順で、既存の行には触れません。テープの
  ように振る舞うものはこちらです。
- **`write`** はテーブルの中身を渡したデータで置き換えます。ただし*新しいバージョン*
  としてで、履歴はすべて残ります。ユニバースの構成銘柄、シンボルマッピング、リスク
  リミットのように、丸ごと言い直される参照データに使います。

どちらも破壊的ではありません。古いバージョンは `db.read(..., version=n)` と、SQL の
`h5i('table', n)` からいつでも読めます。

In [8]:
universe_schema = pa.schema(
    [pa.field("ts", pa.timestamp("us", tz="UTC"), nullable=False), pa.field("symbol", pa.string())]
)
db.create_table("universe", universe_schema, time_column="ts")

asof = pa.scalar(pd.Timestamp("2026-06-01", tz="UTC"), type=pa.timestamp("us", tz="UTC"))
db.write(
    "universe",
    pa.table({"ts": pa.array([asof] * 3), "symbol": pa.array(["AAPL", "MSFT", "NVDA"])}),
    note="June universe",
)
db.write(
    "universe",
    pa.table({"ts": pa.array([asof] * 4), "symbol": pa.array(["AAPL", "MSFT", "NVDA", "AVGO"])}),
    note="June universe, AVGO added",
)
print("head :", db.read("universe")["symbol"].to_pylist())
print("v1   :", db.read("universe", version=1)["symbol"].to_pylist())
[{k: v[k] for k in ("sequence", "op", "rows", "note") if k in v} for v in db.versions("universe")]

head : ['AAPL', 'MSFT', 'NVDA', 'AVGO']
v1   : ['AAPL', 'MSFT', 'NVDA']


[{'sequence': 0, 'op': 'create', 'rows': 0},
 {'sequence': 1, 'op': 'write', 'rows': 3, 'note': 'June universe'},
 {'sequence': 2,
  'op': 'write',
  'rows': 4,
  'note': 'June universe, AVGO added'}]

## `expected_version` による楽観ロック

2つのローダーが1つのテーブルを共有していると、「いつでも好きに append する」やり方は
納品を静かに混ぜてしまいます。`append(..., expected_version=n)` は compare-and-swap
です。テーブルの先頭がまだバージョン `n` のときだけコミットが着地し、そうでなければ
`ConflictError` が上がります。`retryable=True` と、復旧手順を書いたヒントが付いている
はずです。リトライは機械的で、先頭を読み直してもう一度 append するだけです。

In [9]:
day6 = by_day[sessions[5]]
try:
    db.append("trades", day6, expected_version=1)  # stale: head is already at v5
except h5i_db.ConflictError as e:
    print(f"code      {e.code}")
    print(f"retryable {e.retryable}")
    print(f"hint      {e.hint}")

# retry pattern: re-read the head version, then re-append against it
head = db.versions("trades")[-1]["sequence"]
commit = db.append("trades", day6, expected_version=head, note="day 6: CAS append")
print(f"\nretried against v{head} -> committed v{commit['sequence']}")

code      version_conflict
retryable True
hint      re-read the head of "trades" and retry against it; pure appends rebase safely (the CLI and Python bindings already auto-retry those)

retried against v5 -> committed v6


## バッチ化とコンパクション

コミットのたびにマニフェストが1つと、少なくとも1つのセグメントが書かれます。だから
コミットは*まとまり*で打ってください（1日ぶん、1時間ぶん、数千行ぶん）。1行ずつは
禁物です。ただ、1日サイズのコミットでも小さなセグメントは溜まっていき、クエリの
プランニングはそのすべてに触れます。下の「日次ループ → compact」のパターンが通常の
リズムです。平日は小さな append コミットを重ね、`compact` でセグメントを1つにまとめます。
コンパクション自体もただのコミットで、データは同じ、セグメントは減り、履歴は丸ごと
残ります。

In [10]:
for d in sessions[6:]:
    commit = db.append("trades", by_day[d], note=f"daily load {d}")
    print(f"v{commit['sequence']}: +{len(by_day[d]):>6,} rows "
          f"-> {commit['segments_total']:>2} segments total")

v7: +13,806 rows ->  7 segments total
v8: +12,169 rows ->  8 segments total
v9: +11,128 rows ->  9 segments total
v10: +13,093 rows -> 10 segments total
v11: +14,464 rows -> 11 segments total


In [11]:
before = db.versions("trades")[-1]
commit = db.compact("trades")
print(f"compacted: {before['segments']} segments -> {commit['segments_total']}, "
      f"rows unchanged: {commit['rows_total']:,}")

[
    {k: v[k] for k in ("sequence", "op", "rows", "segments", "note") if k in v}
    for v in db.versions("trades")
]

compacted: 11 segments -> 1, rows unchanged: 141,301


[{'sequence': 0, 'op': 'create', 'rows': 0, 'segments': 0},
 {'sequence': 1,
  'op': 'append',
  'rows': 11418,
  'segments': 1,
  'note': 'day 1: arrow feed'},
 {'sequence': 2,
  'op': 'append',
  'rows': 25773,
  'segments': 2,
  'note': 'day 2: pandas'},
 {'sequence': 3,
  'op': 'append',
  'rows': 38842,
  'segments': 3,
  'note': 'day 3: polars'},
 {'sequence': 4,
  'op': 'append',
  'rows': 49876,
  'segments': 4,
  'note': 'day 4: parquet drop'},
 {'sequence': 5,
  'op': 'append',
  'rows': 63593,
  'segments': 5,
  'note': 'day 5: legacy csv'},
 {'sequence': 6,
  'op': 'append',
  'rows': 76641,
  'segments': 6,
  'note': 'day 6: CAS append'},
 {'sequence': 7,
  'op': 'append',
  'rows': 90447,
  'segments': 7,
  'note': 'daily load 2026-06-09'},
 {'sequence': 8,
  'op': 'append',
  'rows': 102616,
  'segments': 8,
  'note': 'daily load 2026-06-10'},
 {'sequence': 9,
  'op': 'append',
  'rows': 113744,
  'segments': 9,
  'note': 'daily load 2026-06-11'},
 {'sequence': 10,
  'op

In [12]:
# One tape, five formats, eleven commits - and SQL sees a single clean table.
db.sql(
    """
    SELECT time_bucket('1d', ts) AS session, count(*) AS trades,
           round(sum(price * size) / 1e6, 1) AS notional_mm
    FROM trades GROUP BY session ORDER BY session
    """
).to_pandas()

,session,trades,notional_mm
0,2026-06-01 00:00:00+00:00,11418,250.4
1,2026-06-02 00:00:00+00:00,14355,339.6
2,2026-06-03 00:00:00+00:00,13069,295.4
3,2026-06-04 00:00:00+00:00,11034,252.4
4,2026-06-05 00:00:00+00:00,13717,309.5
5,2026-06-08 00:00:00+00:00,13048,294.2
6,2026-06-09 00:00:00+00:00,13806,319.5
7,2026-06-10 00:00:00+00:00,12169,264.3
8,2026-06-11 00:00:00+00:00,11128,251.3
9,2026-06-12 00:00:00+00:00,13093,288.0


## まとめ

- Arrow の形をしたものはそのまま append できます。境界でつまずくのは、pandas の
  ナノ秒（`from_pandas(schema=...)`）、polars の `large_string`（`.cast(schema)`）、
  そして型を忘れる CSV（`ConvertOptions`）の3つです。
- `append` はフィードを伸ばす操作（厳密に時刻順）、`write` は中身を新しいバージョンとして
  言い直す操作。どちらも履歴を壊しません。
- コミット辞書（`sequence`、`rows_total`、`segments_total`）は取り込みの受領証です。
  ログに残しましょう。
- `expected_version` は append を compare-and-swap に変えます。`ConflictError`
  （リトライ可）が出たら、先頭を読み直してもう一度 append します。
- コミットはまとめて打ち、小さな append が続いたあとに `compact` します。コンパクションも
  ただのバージョンで、履歴に触れずセグメントだけを併合します。

In [13]:
db.close()